### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this , you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [1]:
!pip install langchain

In [2]:
### Open AI API Key and Open Source models--Llama3,Gemma2,mistral--Groq


In [4]:
import os 
from dotenv import load_dotenv 
load_dotenv()

import openai 
openai.api_key = os.getenv("OPENAI_API_KEY")
groq_api_key=os.getenv("GROQ_API_KEY")


In [5]:
from langchain_groq import ChatGroq 
from langchain_openai import ChatOpenAI

In [ ]:
# model=ChatGroq(model="Gemma2-9b-It", groq_api_key=groq_api_key)
# model


ChatGroq(output_version=None, profile={}, client=<groq.resources.chat.completions.Completions object at 0x13544d1e0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x13544f610>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
# model = ChatGroq(model="llama3-70b-8192", groq_api_key=groq_api_key)
# model

ChatGroq(output_version=None, profile={'max_input_tokens': 8192, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x134d0b6a0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x134d0afb0>, model_name='llama3-70b-8192', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [12]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",  # active model
    groq_api_key=groq_api_key
)

model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x13653f4f0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1365a7a00>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [14]:
from langchain_core.messages import HumanMessage, SystemMessage

messages =[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello how are you?")
]

result=model.invoke(messages)

In [15]:
result

AIMessage(content='Bonjour, comment vas-tu? (informal) or \nBonjour, comment allez-vous? (formal) \n\n(I\'ll assume you\'re using the informal version since you didn\'t specify otherwise)\n\nHowever, if you\'d like I can respond in a more friendly tone: \n\nBonjour ! Comment ça va ? \n\nWhich all roughly translate to "Hello, how are you?"', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 27, 'total_tokens': 101, 'completion_time': 0.167455269, 'completion_tokens_details': None, 'prompt_time': 0.000141958, 'prompt_tokens_details': None, 'queue_time': 0.017200231, 'total_time': 0.167597227}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_2f97044c6d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dd95d-a4d4-7ef1-a26c-450ec297719c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 27, 'output_tokens': 74, 'total_

In [17]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'Bonjour, comment vas-tu? (informal) or \nBonjour, comment allez-vous? (formal) \n\n(I\'ll assume you\'re using the informal version since you didn\'t specify otherwise)\n\nHowever, if you\'d like I can respond in a more friendly tone: \n\nBonjour ! Comment ça va ? \n\nWhich all roughly translate to "Hello, how are you?"'

In [18]:
###Using LCEL - chain the components
chain = model|parser
chain.invoke(messages)

'Bonjour, comment allez-vous?'

In [19]:
###Prompt templates
from langchain_core.prompts import ChatPromptTemplate

generic_template="Translate the following into {language}:"
prompt = ChatPromptTemplate.from_messages(
    [("system", generic_template), ("user","{text}")]
)

In [21]:
result=prompt.invoke({"language":"French", "text":"Hello"})

In [22]:
result

ChatPromptValue(messages=[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})])

In [23]:
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [ ]:
##Chaining together components with LCEL
chain = prompt|model|parser

In [25]:
chain.invoke({"language":"French", "text":"hello"})

'Bonjour!'